In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
print("ML setup ready ✅")


ML setup ready ✅


In [1]:
import pandas as pd

# Load the yield dataset
yield_df = pd.read_csv(r"C:\Users\anjal\OneDrive\Desktop\yield_data.csv")

# Step 1: Look at first few rows
yield_df.head()


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40
1,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40
2,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74
3,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64
4,Andaman and Nicobar Island,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75


In [6]:
import pandas as pd

# 1️⃣ Load the dataset
yield_df = pd.read_csv(r"C:\Users\anjal\OneDrive\Desktop\yield_data.csv")

# 2️⃣ Clean column names: strip spaces and lowercase
yield_df.columns = yield_df.columns.str.strip().str.lower()
print("Columns after cleaning:", yield_df.columns)

# 3️⃣ Standardize State and District names
yield_df['state'] = yield_df['state'].str.strip().str.lower()
yield_df['district'] = yield_df['district'].str.strip().str.lower()

# 4️⃣ Clean Season names
yield_df['season'] = yield_df['season'].astype(str).str.strip()

# 5️⃣ Map Season to Month (manual mapping for unmapped seasons)
season_to_month = {
    'Kharif': 8,    # August
    'Rabi': 2,      # February
    'Autumn': 10,   # October
    'Summer': 5,    # May
    'Whole Year': 6, # Optional mapping
    'Winter': 1      # Optional mapping
}
yield_df['month'] = yield_df['season'].map(season_to_month)

# 6️⃣ Check and print unmapped seasons
unmapped_seasons = yield_df[yield_df['month'].isnull()]['season'].unique()
if len(unmapped_seasons) > 0:
    print("Unmapped seasons remaining:", unmapped_seasons)

# 7️⃣ Handle missing crop values
yield_df['crop'] = yield_df['crop'].fillna(yield_df['crop'].mode()[0])

# 8️⃣ Remove rows with critical missing fields
yield_df.dropna(subset=['state', 'district', 'yield'], inplace=True)

# 9️⃣ Fill missing numeric fields
# a) Production: fill with median per district & crop
yield_df['production'] = yield_df.groupby(['district','crop'])['production'].transform(
    lambda x: x.fillna(x.median())
)
# b) Area: fill missing with global median
yield_df['area'] = yield_df['area'].fillna(yield_df['area'].median())

# 10️⃣ Remove rows with zero or negative yields or productions
yield_df = yield_df[(yield_df['yield'] > 0) & (yield_df['production'] > 0)]

# 11️⃣ Optional: Handle extreme outliers in yield/production (capping)
q_low, q_high = 0.01, 0.99
yield_df['yield'] = yield_df['yield'].clip(lower=yield_df['yield'].quantile(q_low),
                                           upper=yield_df['yield'].quantile(q_high))
yield_df['production'] = yield_df['production'].clip(lower=yield_df['production'].quantile(q_low),
                                                     upper=yield_df['production'].quantile(q_high))

# 12️⃣ Sanity checks
print("Negative/zero yields remaining:", yield_df[yield_df['yield'] <= 0].shape[0])
print("Negative/zero production remaining:", yield_df[yield_df['production'] <= 0].shape[0])
print("Duplicate rows:", yield_df.duplicated().sum())
print("Missing values per column:\n", yield_df.isnull().sum())
print("Unique seasons after mapping:", yield_df['season'].unique())
print("Month coverage:", yield_df['month'].unique())

# 13️⃣ Preview
print(yield_df.head())
print(yield_df.describe())

# 14️⃣ Save fully cleaned yield dataset
yield_df.to_csv(r"C:\Users\anjal\crop-yield-ml\data\yield_cleaned_ready.csv", index=False)
print("✅ Fully cleaned and ready yield data saved successfully!")






Columns after cleaning: Index(['state', 'district', 'crop', 'crop_year', 'season', 'area',
       'production', 'yield'],
      dtype='object')


c:\Users\anjal\crop-yield-ml\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\anjal\crop-yield-ml\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\anjal\crop-yield-ml\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\anjal\crop-yield-ml\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\anjal\crop-yield-ml\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\anjal\crop-yield-ml\.venv\Lib\site-packages\numpy\lib\_nanfunctions_imp

Negative/zero yields remaining: 0
Negative/zero production remaining: 0
Duplicate rows: 0
Missing values per column:
 state         0
district      0
crop          0
crop_year     0
season        0
area          0
production    0
yield         0
month         0
dtype: int64
Unique seasons after mapping: ['Kharif' 'Rabi' 'Autumn' 'Summer' 'Whole Year' 'Winter']
Month coverage: [ 8  2 10  5  6  1]
                        state  district      crop  crop_year  season    area  \
0  andaman and nicobar island  nicobars  Arecanut       2007  Kharif  2439.6   
1  andaman and nicobar island  nicobars  Arecanut       2007    Rabi  1626.4   
2  andaman and nicobar island  nicobars  Arecanut       2008  Autumn  4147.0   
3  andaman and nicobar island  nicobars  Arecanut       2008  Summer  4147.0   
4  andaman and nicobar island  nicobars  Arecanut       2009  Autumn  4153.0   

   production  yield  month  
0      3415.0   1.40      8  
1      2277.0   1.40      2  
2      3060.0   0.74     10  


In [10]:
import pandas as pd
import numpy as np

# 1️⃣ Load Excel, skip first extra header row
weather_df = pd.read_excel(r"C:\Users\anjal\OneDrive\Desktop\weather_data.xlsx", skiprows=1)

# 2️⃣ Inspect the first few rows to see column issues
print(weather_df.head())

# 3️⃣ Rename columns properly
# Only if your columns are standard: PARAMETER, YEAR, LAT, LON, JAN ... DEC, ANN
weather_df.columns = ['file','parameter','year','lat','lon',
                      'jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec','ann']

# 4️⃣ Remove the 'file' column
weather_df = weather_df.drop(columns=['file'])

# 5️⃣ Convert monthly columns to numeric, coerce errors
months = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec','ann']
for m in months:
    weather_df[m] = pd.to_numeric(weather_df[m], errors='coerce')  # strings → NaN

# 6️⃣ Drop rows missing essential info
weather_df = weather_df.dropna(subset=['parameter','year','lat','lon'])

# 7️⃣ Melt wide → long format
weather_long = weather_df.melt(
    id_vars=['parameter','year','lat','lon','ann'],
    value_vars=months[:-1],  # skip 'ann' for monthly long format
    var_name='month',
    value_name='value'
)

# 8️⃣ Map month names to numbers
month_map = {m: i+1 for i, m in enumerate(months[:-1])}
weather_long['month'] = weather_long['month'].map(month_map)

# 9️⃣ Standardize parameter names
weather_long['parameter'] = weather_long['parameter'].str.strip().str.lower()

# 🔟 Handle missing monthly values
weather_long['value'] = weather_long.groupby(['parameter','lat','lon','year'])['value'].transform(
    lambda x: x.fillna(x.mean())
)
weather_long['value'] = weather_long['value'].fillna(weather_long['value'].median())

# 1️⃣1️⃣ Placeholder columns for merging later
weather_long['district'] = np.nan
weather_long['state'] = np.nan

# 1️⃣2️⃣ Final checks
print(weather_long.head(10))
print("Missing values per column:\n", weather_long.isnull().sum())
print("Unique parameters:", weather_long['parameter'].unique())
print("Year range:", weather_long['year'].min(), "-", weather_long['year'].max())
print("Month range:", weather_long['month'].min(), "-", weather_long['month'].max())

# 1️⃣3️⃣ Save cleaned dataset
weather_long.to_csv(r"C:\Users\anjal\crop-yield-ml\data\weather_cleaned_ready.csv", index=False)
print("✅ Weather dataset cleaned and saved successfully!")




   POWER_Regional_Monthly_1997_2020 (1).csv -BEGIN HEADER- Unnamed: 2  \
0  POWER_Regional_Monthly_1997_2020 (1).csv      PARAMETER       YEAR   
1  POWER_Regional_Monthly_1997_2020 (1).csv        T2M_MAX       1997   
2  POWER_Regional_Monthly_1997_2020 (1).csv        T2M_MAX       1997   
3  POWER_Regional_Monthly_1997_2020 (1).csv        T2M_MAX       1997   
4  POWER_Regional_Monthly_1997_2020 (1).csv        T2M_MAX       1997   

  Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8  \
0        LAT        LON        JAN        FEB        MAR        APR   
1       10.0      73.75      27.79      27.74      29.16      29.65   
2       10.0     74.375      27.54      27.89      29.23      29.54   
3       10.0       75.0       27.8      28.16      29.51      29.65   
4       10.0     75.625      27.87      29.52      29.84      29.81   

  Unnamed: 9 Unnamed: 10 Unnamed: 11 Unnamed: 12 Unnamed: 13 Unnamed: 14  \
0        MAY         JUN         JUL         AUG         S

In [26]:
import pandas as pd

# 1️⃣ Load Excel file (skip the NASA header row)
weather_path = r"C:\Users\anjal\OneDrive\Desktop\weather_data.xlsx"
weather_df = pd.read_excel(weather_path, skiprows=1)

# 2️⃣ Assign correct column names (18 total)
weather_df.columns = ['file','parameter','year','lat','lon',
                      'jan','feb','mar','apr','may','jun',
                      'jul','aug','sep','oct','nov','dec','ann']

# 3️⃣ Drop the unnecessary "file" column
weather_df = weather_df.drop(columns=['file'])

# 4️⃣ Convert to numeric where possible
for col in ['year','lat','lon','jan','feb','mar','apr','may','jun',
            'jul','aug','sep','oct','nov','dec','ann']:
    weather_df[col] = pd.to_numeric(weather_df[col], errors='coerce')

# 5️⃣ Save cleaned file
weather_df.to_csv("cleaned_weather.csv", index=False)

print("✅ Weather dataset cleaned and saved successfully!")
print("Columns:", list(weather_df.columns))
print("Year range:", weather_df['year'].min(), "-", weather_df['year'].max())


✅ Weather dataset cleaned and saved successfully!
Columns: ['parameter', 'year', 'lat', 'lon', 'jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec', 'ann']
Year range: 1997.0 - 2020.0


In [29]:
# Save to Excel instead of CSV
weather_df.to_excel("cleaned_weather.xlsx", index=False)

print("✅ Cleaned weather data saved as Excel file!")


✅ Cleaned weather data saved as Excel file!


In [ ]:
import os

# Define path where you want to save
save_path = os.path.join(os.path.expanduser("~"), "Desktop", "cleaned_weather_data.csv")

# Save the DataFrame
weather_df.to_csv(save_path, index=False)

print(f"✅ Weather data saved successfully at: {save_path}")


✅ Weather data saved successfully at: C:\Users\anjal\Desktop\cleaned_weather_data.csv


In [ ]:
import pandas as pd
import os

# Suppose your cleaned yield DataFrame is already in memory as yield_df
# Save it to Desktop
yield_save_path = os.path.join(os.path.expanduser("~"), "Desktop", "cleaned_yield_data.csv")
yield_df.to_csv(yield_save_path, index=False)

print(f"✅ Yield data saved successfully at: {yield_save_path}")


✅ Yield data saved successfully at: C:\Users\anjal\Desktop\cleaned_yield_data.csv


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
ś
# ========= 1️⃣ Load datasets =========
soil_df = pd.read_csv(r"C:\Users\anjal\OneDrive\Desktop\soil_data_cleaned.csv")
weather_df = pd.read_csv(r"C:\Users\anjal\OneDrive\Desktop\cleaned_weather.csv")
yield_df = pd.read_csv(r"C:\Users\anjal\OneDrive\Desktop\yield_data.csv")

# ========= 2️⃣ Clean column names =========
soil_df.columns = soil_df.columns.str.strip().str.lower()
weather_df.columns = weather_df.columns.str.strip().str.lower()
yield_df.columns = yield_df.columns.str.strip().str.lower()

# Fix yield dataframe columns
yield_df = yield_df.rename(columns={
    'district ': 'district',
    'area ': 'area',
    'crop_year': 'year'
})

# ========= 3️⃣ Pivot weather =========
weather_pivot = weather_df.pivot_table(
    index=['year','lat','lon'],
    columns='parameter',
    values='ann'   # take annual values
).reset_index()

weather_pivot.columns.name = None  # remove pivot level name

# ========= 4️⃣ Merge soil + weather =========
soil_weather = pd.merge(
    soil_df,
    weather_pivot,
    on=['year','lat','lon'],
    how='left'
)

# ========= 5️⃣ Merge with yield =========
final_merged = pd.merge(
    yield_df,
    soil_weather,
    on='year',
    how='left'
)

# ========= 6️⃣ Save human-readable version =========
final_merged.to_csv(r"C:\Users\anjal\OneDrive\Desktop\merged_dataset.csv", index=False)

# ========= 7️⃣ Encode categorical columns =========
encoded_df = final_merged.copy()
label_encoders = {}

for col in ['state', 'district', 'crop', 'season']:
    if col in encoded_df.columns:
        le = LabelEncoder()
        encoded_df[col] = le.fit_transform(encoded_df[col].astype(str))
        label_encoders[col] = dict(zip(le.classes_, le.transform(le.classes_)))

# ========= 8️⃣ Save encoded version =========
encoded_df.to_csv(r"C:\Users\anjal\OneDrive\Desktop\merged_dataset_encoded.csv", index=False)

print("✅ Merged dataset created successfully!")
print("🔹 Human-readable saved as merged_dataset.csv")
print("🔹 Encoded for ML saved as merged_dataset_encoded.csv")
print("Columns:", list(encoded_df.columns)[:20], "...")ś


C:\Users\anjal\AppData\Local\Temp\ipykernel_24396\2004521424.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  soil_weather = pd.merge(


✅ Merged dataset created successfully!
🔹 Human-readable saved as merged_dataset.csv
🔹 Encoded for ML saved as merged_dataset_encoded.csv
Columns: ['state', 'district', 'crop', 'year', 'season', 'area', 'production', 'yield', 'lat', 'lon', 'month', 'bdod', 'sand', 'silt', 'clay', 'phh2o', 'ocd', 'ALLSKY_SFC_SW_DWN', 'GWETROOT', 'GWETTOP'] ...


In [1]:
import pandas as pd

# --------------------------
# 1. Load datasets
# --------------------------
weather_df = pd.read_excel("weather.xlsx")
soil_df = pd.read_excel("soil.xlsx")
yield_df = pd.read_excel("yield.xlsx")

# --------------------------
# 2. Clean column names
# --------------------------
def clean_cols(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df

weather_df = clean_cols(weather_df)
soil_df = clean_cols(soil_df)
yield_df = clean_cols(yield_df)

print("Weather Columns:", list(weather_df.columns))
print("Soil Columns:", list(soil_df.columns))
print("Yield Columns:", list(yield_df.columns))

# --------------------------
# 3. Standardize district & year
# --------------------------
def normalize_text(df, col):
    df[col] = df[col].astype(str).str.strip().str.lower()
    return df

for df in [weather_df, soil_df, yield_df]:
    if "district" in df.columns:
        normalize_text(df, "district")
    if "year" in df.columns:
        df["year"] = df["year"].astype(int)

# --------------------------
# 4. Aggregate weather data yearly (if monthly/daily exists)
# --------------------------
# Example: avg rainfall, avg temp etc. (adapt as per your columns)
weather_yearly = weather_df.groupby(["district", "year"]).agg({
    "rainfall": "sum",          # yearly rainfall
    "temperature": "mean",      # avg yearly temperature
    "humidity": "mean"          # avg yearly humidity
}).reset_index()

# --------------------------
# 5. Merge datasets step by step
# --------------------------
# Merge soil + yield first
merged = pd.merge(yield_df, soil_df, on=["district", "year"], how="outer")

# Add weather data
final_df = pd.merge(merged, weather_yearly, on=["district", "year"], how="outer")

print("Merged Shape:", final_df.shape)
print("Sample Rows:\n", final_df.head())

# --------------------------
# 6. Handle missing values
# --------------------------
# Fill numeric columns with mean (you can also try median or interpolation)
num_cols = final_df.select_dtypes(include="number").columns
final_df[num_cols] = final_df[num_cols].fillna(final_df[num_cols].mean())

# Fill district/year text if missing
final_df["district"] = final_df["district"].fillna("unknown")
final_df["year"] = final_df["year"].fillna(-1).astype(int)

# --------------------------
# 7. Save cleaned dataset
# --------------------------
final_df.to_csv("final_dataset.csv", index=False)

print("✅ Final dataset saved as final_dataset.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'weather.xlsx'